# P1-E01. — Raw CAUEEG Data, Metadata and Cohort Audit

**Objective 1, Experiment 1**  
**Purpose:** Build the first authoritative audit directly from the downloaded CAUEEG dataset. 

## Summary

The notebook inventories the raw dataset, discovers EEG and metadata files, audits metadata schemas, reads EEG headers without loading full signals, tests participant–recording linkage, identifies exact/possible duplicate files, and creates the initial cohort reconciliation outputs.

No preprocessing, feature extraction, or classification is performed here. Labels are not silently inferred from arbitrary text: candidate label columns are reported and must pass explicit checks before a cohort is locked.

## Context & Methods

### Key assumptions

- Dataset root: `D:\EEG_AD_Research\data\caueeg-dataset\caueeg-dataset`.
- A participant is the audit unit; a participant may have one or more recordings.
- Raw data are never modified.
- The audit writes new outputs to `D:\EEG_AD_Research\paper1_experiments\paper1_outputs\p1_e01_raw_audit`.
- Full file hashes are calculated only for files sharing the same size, reducing unnecessary I/O.

### 1. Setup and publication-output contract

In [ ]:
from pathlib import Path
import json, hashlib, re, sys, platform, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

DATA_ROOT = Path(r"D:\EEG_AD_Research\data\caueeg-dataset\caueeg-dataset")
PROJECT_ROOT = Path(r"D:\EEG_AD_Research")
OUTPUT_ROOT = PROJECT_ROOT / "paper1_experiments" / "paper1_outputs" / "p1_e01_raw_audit"
FIG_DIR, TABLE_DIR = OUTPUT_ROOT/"figures", OUTPUT_ROOT/"tables"
for p in (OUTPUT_ROOT, FIG_DIR, TABLE_DIR): p.mkdir(parents=True, exist_ok=True)

EEG_EXTENSIONS={'.edf','.bdf','.vhdr','.set','.fif','.cnt','.egi','.mff','.npy','.npz','.mat'}
TABULAR_EXTENSIONS={'.csv','.tsv','.xlsx','.xls','.parquet'}
TEXT_METADATA_EXTENSIONS={'.json','.txt'}
CLASS_ORDER=['Normal','MCI','AD']
CLASS_COLORS={'Normal':'#0072B2','MCI':'#E69F00','AD':'#D55E00'}

mpl.rcParams.update({'font.family':'Arial','font.size':9,'axes.titlesize':10,
                     'axes.labelsize':9,'legend.fontsize':8,'savefig.dpi':600,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'pdf.fonttype':42,'ps.fonttype':42})
sns.set_theme(style='ticks',context='paper')

def save_table(df,name,index=False):
    df.to_csv(TABLE_DIR/f'{name}.csv',index=index)
    try: df.to_excel(TABLE_DIR/f'{name}.xlsx',index=index)
    except Exception as exc: warnings.warn(f'Excel export skipped: {exc}')

def save_figure(fig,name,width_mm=183,height_mm=100):
    fig.set_size_inches(width_mm/25.4,height_mm/25.4)
    for ext in ('png','pdf','svg'):
        fig.savefig(FIG_DIR/f'{name}.{ext}',bbox_inches='tight',facecolor='white')

def canonical_id(value):
    s=str(value).strip()
    if not s or s.lower() in {'nan','none'}: return ''
    if re.fullmatch(r'\d+\.0',s): s=s[:-2]
    digits=''.join(re.findall(r'\d+',s))
    return str(int(digits)) if digits else s.lower()

assert DATA_ROOT.exists(), f'Dataset root does not exist: {DATA_ROOT}'
print('Raw dataset:',DATA_ROOT)
print('Audit output:',OUTPUT_ROOT)

## Data

### 2. Create a complete raw-file inventory

In [ ]:
all_files=sorted(p for p in DATA_ROOT.rglob('*') if p.is_file())
if not all_files: raise RuntimeError('No files found below DATA_ROOT')

inventory=pd.DataFrame([{
    'relative_path':str(p.relative_to(DATA_ROOT)),
    'absolute_path':str(p),
    'extension':p.suffix.lower(),
    'bytes':p.stat().st_size,
    'size_MiB':p.stat().st_size/1024**2,
    'modified_time':pd.Timestamp(p.stat().st_mtime,unit='s').isoformat()
} for p in all_files])
inventory['file_type']=np.select([
    inventory.extension.isin(EEG_EXTENSIONS),
    inventory.extension.isin(TABULAR_EXTENSIONS),
    inventory.extension.isin(TEXT_METADATA_EXTENSIONS)
],['candidate_EEG','tabular_metadata','text_metadata'],default='other')
save_table(inventory,'01_complete_raw_file_inventory')
display(inventory.file_type.value_counts().rename('files').to_frame())
display(inventory.groupby('extension',dropna=False).agg(files=('relative_path','size'),GiB=('bytes',lambda x:x.sum()/1024**3)).sort_values('GiB',ascending=False).head(30))

In [ ]:
summary=(inventory.groupby('file_type',as_index=False)
         .agg(files=('relative_path','size'),total_GiB=('bytes',lambda x:x.sum()/1024**3)))
fig,ax=plt.subplots()
sns.barplot(data=summary,x='file_type',y='files',color='#2C7FB8',ax=ax)
ax.set(xlabel='',ylabel='Number of files',title='CAUEEG raw-data inventory by file category')
ax.tick_params(axis='x',rotation=20)
for c in ax.containers: ax.bar_label(c,fmt='%d',padding=2)
save_figure(fig,'figure_1_raw_file_inventory',183,95); plt.show()

### 3. Audit readable metadata schemas without choosing labels

In [ ]:
def read_table_safe(path):
    ext=path.suffix.lower()
    if ext=='.csv': return pd.read_csv(path,low_memory=False)
    if ext=='.tsv': return pd.read_csv(path,sep='\t',low_memory=False)
    if ext in {'.xlsx','.xls'}: return pd.read_excel(path)
    if ext=='.parquet': return pd.read_parquet(path)
    raise ValueError(ext)

schema_rows=[]; tables={}
for p in [Path(x) for x in inventory.loc[inventory.file_type.eq('tabular_metadata'),'absolute_path']]:
    try:
        frame=read_table_safe(p); tables[str(p)]=frame
        schema_rows.append({'relative_path':str(p.relative_to(DATA_ROOT)),'status':'readable',
                            'rows':len(frame),'columns':len(frame.columns),
                            'column_names':' | '.join(map(str,frame.columns))})
    except Exception as exc:
        schema_rows.append({'relative_path':str(p.relative_to(DATA_ROOT)),'status':'FAILED',
                            'rows':np.nan,'columns':np.nan,'column_names':'','error':repr(exc)})
metadata_schema=pd.DataFrame(schema_rows)
save_table(metadata_schema,'02_tabular_metadata_schema_audit')
display(metadata_schema)

In [ ]:
ID_TOKENS=('participant','subject','patient','case','record','id')
LABEL_TOKENS=('diagnosis','diagnostic','label','group','class','disease','dementia','mci','normal')
AGE_TOKENS=('age','years')
SEX_TOKENS=('sex','gender')

candidate_rows=[]
for path,frame in tables.items():
    cols=list(map(str,frame.columns))
    candidate_rows.append({
        'relative_path':str(Path(path).relative_to(DATA_ROOT)),
        'candidate_id_columns':' | '.join(c for c in cols if any(t in c.lower() for t in ID_TOKENS)),
        'candidate_label_columns':' | '.join(c for c in cols if any(t in c.lower() for t in LABEL_TOKENS)),
        'candidate_age_columns':' | '.join(c for c in cols if any(t in c.lower() for t in AGE_TOKENS)),
        'candidate_sex_columns':' | '.join(c for c in cols if any(t in c.lower() for t in SEX_TOKENS))
    })
candidates=pd.DataFrame(candidate_rows)
save_table(candidates,'03_candidate_metadata_fields')
display(candidates)
print('Review this table before setting authoritative metadata fields in Section 7.')

### 4. Inspect JSON structure, including annotation files

In [ ]:
json_rows=[]
for p in [Path(x) for x in inventory.loc[inventory.extension.eq('.json'),'absolute_path']]:
    try:
        obj=json.loads(p.read_text(encoding='utf-8',errors='replace'))
        if isinstance(obj,dict): keys=list(obj.keys())[:100]; n=len(obj)
        elif isinstance(obj,list): keys=list(obj[0].keys())[:100] if obj and isinstance(obj[0],dict) else []; n=len(obj)
        else: keys=[]; n=1
        json_rows.append({'relative_path':str(p.relative_to(DATA_ROOT)),'status':'readable',
                          'root_type':type(obj).__name__,'root_length':n,'sample_keys':' | '.join(map(str,keys))})
    except Exception as exc:
        json_rows.append({'relative_path':str(p.relative_to(DATA_ROOT)),'status':'FAILED',
                          'root_type':'','root_length':np.nan,'sample_keys':'','error':repr(exc)})
json_audit=pd.DataFrame(json_rows)
save_table(json_audit,'04_json_structure_audit')
display(json_audit)

## Results

### 5. Identify exact and possible duplicate raw files

In [ ]:
same_size=inventory[inventory.duplicated('bytes',keep=False) & inventory.bytes.gt(0)].copy()
def file_sha256(path,block=4*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(block),b''): h.update(chunk)
    return h.hexdigest()

if len(same_size):
    same_size['sha256']=[file_sha256(p) for p in same_size.absolute_path]
    exact_duplicates=same_size[same_size.duplicated('sha256',keep=False)].sort_values(['sha256','relative_path'])
else:
    same_size['sha256']=pd.Series(dtype=str); exact_duplicates=same_size.copy()
save_table(same_size,'05_same_size_file_candidates')
save_table(exact_duplicates,'06_confirmed_exact_file_duplicates')
print('Same-size candidates:',len(same_size),'Confirmed duplicate rows:',len(exact_duplicates))
display(exact_duplicates[['relative_path','bytes','sha256']].head(50))

### 6. Audit EEG headers without loading full signals

In [ ]:
try:
    import mne
except ImportError as exc:
    raise ImportError('Install MNE in eeg_snn first: conda install -n eeg_snn -c conda-forge mne --solver=libmamba -y') from exc

def read_raw_header(path):
    ext=path.suffix.lower(); kwargs=dict(preload=False,verbose='ERROR')
    if ext in {'.edf','.bdf'}: return mne.io.read_raw_edf(path,**kwargs)
    if ext=='.fif': return mne.io.read_raw_fif(path,**kwargs)
    if ext=='.vhdr': return mne.io.read_raw_brainvision(path,**kwargs)
    if ext=='.set': return mne.io.read_raw_eeglab(path,**kwargs)
    if ext=='.cnt': return mne.io.read_raw_cnt(path,**kwargs)
    if ext in {'.egi','.mff'}: return mne.io.read_raw_egi(path,**kwargs)
    raise NotImplementedError(f'Header reader not defined for {ext}')

eeg_candidates=inventory[inventory.file_type.eq('candidate_EEG')].copy()
header_rows=[]
for number,p in enumerate(map(Path,eeg_candidates.absolute_path),1):
    row={'relative_path':str(p.relative_to(DATA_ROOT)),'extension':p.suffix.lower(),'bytes':p.stat().st_size}
    try:
        raw=read_raw_header(p)
        row.update(status='readable',sfreq_Hz=float(raw.info['sfreq']),n_channels=len(raw.ch_names),
                   n_samples=int(raw.n_times),duration_seconds=float(raw.n_times/raw.info['sfreq']),
                   channel_names=' | '.join(raw.ch_names),n_annotations=len(raw.annotations))
        raw.close()
    except NotImplementedError as exc:
        row.update(status='requires_format_specific_reader',error=str(exc))
    except Exception as exc:
        row.update(status='FAILED',error=repr(exc))
    header_rows.append(row)
    if number%100==0: print(f'Header-audited {number}/{len(eeg_candidates)}')

headers=pd.DataFrame(header_rows)
save_table(headers,'07_eeg_header_audit')
display(headers.status.value_counts(dropna=False).rename('recordings').to_frame())
display(headers[headers.status.eq('readable')][['sfreq_Hz','n_channels','duration_seconds']].describe().T)

In [ ]:
readable=headers[headers.status.eq('readable')].copy()
if len(readable):
    fig,axes=plt.subplots(1,3,figsize=(7.2,2.8))
    sns.histplot(readable.sfreq_Hz,discrete=True,ax=axes[0],color='#0072B2'); axes[0].set_title('Sampling rate'); axes[0].set_xlabel('Hz')
    sns.histplot(readable.n_channels,discrete=True,ax=axes[1],color='#E69F00'); axes[1].set_title('Channel count')
    sns.histplot(readable.duration_seconds/60,ax=axes[2],color='#009E73'); axes[2].set_title('Recording duration'); axes[2].set_xlabel('Minutes')
    save_figure(fig,'figure_2_eeg_header_distributions',183,75); plt.show()

### 7. Configure and lock authoritative metadata fields

In [ ]:
# Complete these only after reviewing tables 02–04 above.
# Example:
# AUTHORITATIVE_METADATA_FILE = DATA_ROOT / 'annotation.xlsx'
# ID_COLUMN = 'patient_id'
# LABEL_COLUMN = 'diagnosis'
# AGE_COLUMN = 'age'
# SEX_COLUMN = 'sex'
AUTHORITATIVE_METADATA_FILE = None
ID_COLUMN = None
LABEL_COLUMN = None
AGE_COLUMN = None
SEX_COLUMN = None

LABEL_MAP={
    'normal':'Normal','control':'Normal','healthy control':'Normal','hc':'Normal','nc':'Normal',
    'mci':'MCI','mild cognitive impairment':'MCI',
    'ad':'AD','alzheimer':'AD',"alzheimer's disease":'AD','alzheimer disease':'AD'
}

if AUTHORITATIVE_METADATA_FILE is None:
    print('CONFIGURATION REQUIRED — review candidate metadata fields, set the five variables, then rerun this cell.')
    cohort=None
else:
    source=read_table_safe(Path(AUTHORITATIVE_METADATA_FILE))
    required=[ID_COLUMN,LABEL_COLUMN]
    missing=[c for c in required if c not in source.columns]
    if missing: raise ValueError(f'Configured columns absent: {missing}; available={source.columns.tolist()}')
    cohort=pd.DataFrame({'participant_id':source[ID_COLUMN].map(canonical_id),
                         'label_raw':source[LABEL_COLUMN].astype(str).str.strip()})
    cohort['group']=cohort.label_raw.str.lower().map(LABEL_MAP)
    if AGE_COLUMN: cohort['age']=pd.to_numeric(source[AGE_COLUMN],errors='coerce')
    if SEX_COLUMN: cohort['sex']=source[SEX_COLUMN].astype(str).str.strip()
    if cohort.participant_id.eq('').any(): raise ValueError('Blank participant IDs in authoritative metadata')
    if cohort.participant_id.duplicated().any():
        display(cohort[cohort.participant_id.duplicated(False)].sort_values('participant_id'))
        raise ValueError('Duplicate participant IDs require reconciliation before cohort locking')
    unmapped=cohort[cohort.group.isna()].label_raw.value_counts()
    if len(unmapped):
        display(unmapped.rename('participants').to_frame())
        raise ValueError('Unmapped labels found. Verify terminology and update LABEL_MAP explicitly.')
    save_table(cohort,'08_authoritative_participant_metadata')
    display(cohort.group.value_counts().reindex(CLASS_ORDER,fill_value=0).rename('participants').to_frame())

### 8. Link participant metadata to raw EEG recordings

In [ ]:
def candidate_id_from_path(relative_path):
    # Extract digit tokens but do not presume which token is the participant ID.
    tokens=re.findall(r'\d+',str(relative_path))
    return canonical_id(tokens[-1]) if tokens else ''

recordings=headers.copy()
recordings['path_candidate_id']=recordings.relative_path.map(candidate_id_from_path)
if cohort is None:
    linkage=pd.DataFrame(columns=['participant_id','recordings','readable_recordings'])
    print('Participant linkage deferred until Section 7 is configured.')
else:
    known=set(cohort.participant_id)
    recordings['participant_id']=recordings.path_candidate_id.where(recordings.path_candidate_id.isin(known),'')
    linkage=(recordings[recordings.participant_id.ne('')].groupby('participant_id',as_index=False)
             .agg(recordings=('relative_path','size'),readable_recordings=('status',lambda s:s.eq('readable').sum())))
    reconciliation=(cohort.merge(linkage,on='participant_id',how='left')
                    .fillna({'recordings':0,'readable_recordings':0}))
    reconciliation[['recordings','readable_recordings']]=reconciliation[['recordings','readable_recordings']].astype(int)
    save_table(recordings,'09_recording_index_with_candidate_participant_ids')
    save_table(reconciliation,'10_participant_recording_reconciliation')
    display(reconciliation.groupby('group').agg(participants=('participant_id','size'),
             without_recording=('recordings',lambda x:x.eq(0).sum()),
             with_multiple_recordings=('recordings',lambda x:x.gt(1).sum()),
             without_readable_recording=('readable_recordings',lambda x:x.eq(0).sum())))

### 9. Create the initial audit decision and participant-flow figure

In [ ]:
checks=[
    {'check':'raw dataset directory exists','pass':DATA_ROOT.exists(),'detail':str(DATA_ROOT)},
    {'check':'raw files discovered','pass':len(inventory)>0,'detail':len(inventory)},
    {'check':'candidate EEG files discovered','pass':len(eeg_candidates)>0,'detail':len(eeg_candidates)},
    {'check':'metadata tables inventoried','pass':len(metadata_schema)>0,'detail':len(metadata_schema)},
    {'check':'authoritative metadata configured','pass':cohort is not None,'detail':'required before cohort lock'},
]
if cohort is not None:
    checks.extend([
        {'check':'unique participant IDs','pass':not cohort.participant_id.duplicated().any(),'detail':len(cohort)},
        {'check':'all labels mapped','pass':cohort.group.notna().all(),'detail':cohort.group.value_counts().to_dict()},
        {'check':'all participants linked to recording','pass':reconciliation.recordings.gt(0).all(),
         'detail':int(reconciliation.recordings.eq(0).sum())},
        {'check':'all participants linked to readable recording','pass':reconciliation.readable_recordings.gt(0).all(),
         'detail':int(reconciliation.readable_recordings.eq(0).sum())}
    ])
checks=pd.DataFrame(checks)
save_table(checks,'11_raw_audit_checks')
display(checks)

if cohort is not None:
    counts=cohort.group.value_counts().reindex(CLASS_ORDER,fill_value=0)
    fig,ax=plt.subplots()
    bars=ax.bar(counts.index,counts.values,color=[CLASS_COLORS[x] for x in counts.index])
    ax.bar_label(bars,padding=3); ax.set(xlabel='Diagnostic group',ylabel='Participants',
        title='Metadata-verified CAUEEG participants before eligibility exclusions')
    save_figure(fig,'figure_3_metadata_verified_class_counts',120,90); plt.show()

### 10. Save reproducibility record and final status

In [ ]:
run_record={
    'experiment':'P1-E01.2 raw CAUEEG audit',
    'timestamp_utc':pd.Timestamp.utcnow().isoformat(),
    'data_root':str(DATA_ROOT),'raw_file_count':len(inventory),
    'candidate_eeg_files':len(eeg_candidates),'readable_eeg_headers':int(headers.status.eq('readable').sum()),
    'metadata_configured':cohort is not None,
    'python':sys.version,'platform':platform.platform(),
    'status':'AUDIT INVENTORY COMPLETE — configure authoritative metadata before cohort lock' if cohort is None
             else ('RAW AUDIT PASSED' if checks['pass'].all() else 'REVIEW FLAGGED CHECKS BEFORE COHORT LOCK')
}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))

## Takeaways

This notebook intentionally stops short of preprocessing or classification. The outputs establish what raw data exist, what metadata fields are available, whether recording headers are readable, whether participant IDs link correctly, and which issues must be resolved before defining the cohort.

.